In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import year, col, array, regexp_extract, mean, split, regexp_replace
from pyspark.sql.types import StringType, IntegerType, TimeType, ArrayType
from functools import reduce
spark = SparkSession.builder.appName("Movies1_testing").getOrCreate()
movies1_df = spark.read.table("databricks_udemy.default.imdb_top_1000")
display(movies1_df.limit(10))

In [0]:
movies1_df.printSchema()

Changing Data Types

In [0]:
movies1_df = movies1_df.withColumn('Genre',regexp_replace(col("Genre"), " ", ""))
display(movies1_df.limit(10))

In [0]:
movies1_df = movies1_df.withColumn('Genre', split(col('Genre'), ',').cast(ArrayType(StringType())))
movies1_df.printSchema()

In [0]:
movies1_df = movies1_df.withColumn('Stars', array(col('Star1'),col('Star2'),col('Star3'),col('Star4')))
movies1_df = movies1_df.drop('Star1','Star2','Star3','Star4')
display(movies1_df.limit(10))

In [0]:
movies1_df = movies1_df.withColumn('Runtime', regexp_extract('Runtime', r"(\d+)",1).cast(IntegerType()))
movies1_df.printSchema()

In [0]:
bad_rows=movies1_df.filter(~col('Released_Year').rlike('^[0-9]+$'))
movies1_df = movies1_df.subtract(bad_rows)
display(movies1_df.limit(10))
display(bad_rows)

In [0]:
movies1_df= movies1_df.withColumn('Released_Year', movies1_df['Released_Year'].cast(IntegerType()))
movies1_df.printSchema()

In [0]:
movies1_df = movies1_df.withColumn('Gross',regexp_replace(col('Gross'), ',', ''))
display(movies1_df.limit(10))

In [0]:
movies1_df = movies1_df.withColumn('Gross', movies1_df['Gross'].cast(IntegerType()))
movies1_df.printSchema()

In [0]:
display(movies1_df.filter(col('Gross').isNull()))

In [0]:
display(movies1_df.where(reduce(lambda x, y: x | y, (col(x).isNull() for x in movies1_df.columns))))

In [0]:
movies1_df.filter(col('Certificate').isNull()).count()



In [0]:
movies1_df.filter(col('Gross').isNull()).count()

In [0]:
movies1_df.filter(col('Meta_score').isNull()).count()

In [0]:
movies1_df = movies1_df.fillna({'Gross': '0'})
movies1_df = movies1_df.fillna({'Certificate': 'Unknown'})
avg_meta = movies1_df.where(col('Meta_score').isNotNull()).select(mean(col('Meta_score')))
movies1_df = movies1_df.fillna({'Meta_score': avg_meta.collect()[0][0]}) ## because just collect() returns [Row(avg(Meta_score)=number)] as a list

In [0]:
movies1_df.filter(col('Meta_score').isNull()).count()

In [0]:
movies1_df.filter(col('Gross').isNull()).count()

In [0]:
movies1_df.filter(col('Certificate').isNull()).count()

In [0]:
movies_cleaned = movies1_df

In [0]:
spark.sql('USE imdb')

In [0]:
spark.sql("""CREATE TABLE IF NOT EXISTS movies1 (
    Poster_Link STRING,
    Series_Title STRING,
    Released_Year INTEGER,
    Certificate STRING,
    Runtime INTEGER,
    Genre ARRAY<STRING>,
    IMDB_Rating DOUBLE,
    Overview STRING,
    Meta_score LONG,
    Director STRING,
    No_of_Votes LONG,
    Gross INTEGER,
    Stars ARRAY<STRING>
)""")

In [0]:
movies_cleaned.createOrReplaceTempView('movies1_tmp_table')

In [0]:
spark.sql('describe movies1').show()

In [0]:
spark.sql('describe movies1_tmp_table').show()

In [0]:
spark.sql("""INSERT OVERWRITE imdb.movies1
          SELECT `Poster_link`,
                `Series_Title`,
                `Released_Year`,
                `Certificate`,
                `Runtime`,
                `Genre`,
                `IMDB_Rating`,
                `Overview`,
                `Meta_score`,
                `Director`,
                `No_of_Votes`,
                `Gross`,
                `Stars`
          FROM movies1_tmp_table""")